# Week 3 follow-up #2: full dataset + discriminative LR + per-epoch checkpoints (Kaggle GPU)

**Why this run exists:** configs D/E (unfreeze last 2 vision layers, 3 epochs, same 1500-image
subset, same LR for vision and decoder) overfit -- train loss dropped 1.99 -> 0.85 but BLEU
dropped 27.2 -> 23.4 on held-out images. That's a training-recipe problem (too little data for
the added capacity, LR too high for the pretrained vision weights, too many epochs), not
evidence the model needs *more* of the same treatment.

**This run fixes the recipe, not just adds more of it:**
1. **Nearly all the data**: 7,991 training images (vs. 1,500) -- the same 100 eval images stay
   held out, untouched, for a fair comparison against every prior config (A/C/D/E all used the
   same 100).
2. **Discriminative learning rate**: vision layers at 5e-6, decoder at 5e-5 (10x lower for the
   pretrained backbone) -- standard practice to avoid the backbone getting knocked around by a
   head that's still adapting.
3. **Checkpoint + evaluate after every epoch** (3 epochs, both greedy and beam decoding each
   time) -- gives an actual train-loss-vs-eval-BLEU curve instead of committing to an epoch count
   blind. If eval performance peaks at epoch 1 or 2 and then drops, that's directly visible and we
   just use the best epoch's checkpoint.

## Setup
1. New Kaggle Notebook, paste this file in.
2. **Settings -> Accelerator -> GPU** (T4 x2 or P100).
3. **Add Data** -> `adityajn105/flickr8k`.
4. Run all cells. Estimated ~50-65 minutes total (3 epochs x ~15-20 min + 6 eval passes).
5. Download `week3_finetune_fulldata_results.json`/`.csv` from the Output tab. The results will
   tell you which epoch was best -- download only that one checkpoint
   (`config_F_epoch{N}_checkpoint/`) to save time, unless you want to keep all three.

### 1. Setup

In [ ]:
import os
import json
import time
import random
import pandas as pd
import torch
import evaluate
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

KAGGLE_INPUT_DIR = "/kaggle/input/flickr8k"
OUTPUT_DIR = "/kaggle/working"

CAPTIONS_PATH = os.path.join(KAGGLE_INPUT_DIR, "captions.txt")
IMG_DIR = os.path.join(KAGGLE_INPUT_DIR, "Images")

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU detected -- check Settings > Accelerator on Kaggle."
print(f"Using device: {device}")

MODEL_NAME = "Salesforce/blip-image-captioning-base"

df = pd.read_csv(CAPTIONS_PATH)
df.columns = ["image", "caption"]
print(f"Total captions: {len(df)}, unique images: {df['image'].nunique()}")

### 2. Train / eval split -- SAME eval set as every prior config (seed=42, first 100 images),
now training on all the rest instead of a 1500-image slice.

In [ ]:
random.seed(42)
all_images = df["image"].drop_duplicates().tolist()
random.shuffle(all_images)

eval_images = all_images[:100]
train_images = all_images[100:]   # everything else -- ~7991 images

train_df = df[df["image"].isin(train_images)].groupby("image").head(2).reset_index(drop=True)
eval_refs = [df[df["image"] == img]["caption"].tolist() for img in eval_images]

print(f"Training pairs: {len(train_df)} (from {len(train_images)} images)")
print(f"Eval images: {len(eval_images)} (identical to configs A-E)")

### 3. Dataset / collator (batch size raised to 16 for throughput on the larger dataset)

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)

class FlickrFineTuneDataset(Dataset):
    def __init__(self, dataframe, img_dir):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        return image, row["caption"]

def collate_fn(batch):
    images, captions = zip(*batch)
    inputs = processor(
        images=list(images), text=list(captions),
        padding="max_length", truncation=True, max_length=32, return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

train_dataset = FlickrFineTuneDataset(train_df, IMG_DIR)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
print(f"{len(train_loader)} batches/epoch")

### 4. Eval function (unchanged) + discriminative-LR training with per-epoch checkpointing

In [ ]:
@torch.no_grad()
def evaluate_model(model, decoding="greedy"):
    model.eval()
    gen_kwargs = {"max_new_tokens": 30}
    gen_kwargs["num_beams"] = 4 if decoding == "beam" else 1

    predictions = []
    for img_name in tqdm(eval_images, desc=f"eval decoding={decoding}"):
        image = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        out = model.generate(**inputs, **gen_kwargs)
        predictions.append(processor.decode(out[0], skip_special_tokens=True))

    bleu = evaluate.load("sacrebleu").compute(predictions=predictions, references=eval_refs)
    rouge = evaluate.load("rouge").compute(predictions=predictions, references=eval_refs)
    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }, predictions


def train_with_checkpoints(lr_vision, lr_decoder, num_epochs, unfreeze_last_n_vision_layers):
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

    num_vision_layers = len(model.vision_model.encoder.layers)
    freeze_up_to = num_vision_layers - unfreeze_last_n_vision_layers
    for i, layer in enumerate(model.vision_model.encoder.layers):
        for p in layer.parameters():
            p.requires_grad = i >= freeze_up_to
    for p in model.vision_model.embeddings.parameters():
        p.requires_grad = False

    vision_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("vision_model")]
    other_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("vision_model")]
    print(f"Vision params trainable: {sum(p.numel() for p in vision_params):,} @ lr={lr_vision}")
    print(f"Other (decoder) params trainable: {sum(p.numel() for p in other_params):,} @ lr={lr_decoder}")

    optimizer = torch.optim.AdamW([
        {"params": vision_params, "lr": lr_vision},
        {"params": other_params, "lr": lr_decoder},
    ])

    epoch_results = {}
    for epoch in range(num_epochs):
        model.train()
        losses = []
        for batch in tqdm(train_loader, desc=f"train epoch {epoch + 1}/{num_epochs}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())

        metrics_greedy, _ = evaluate_model(model, decoding="greedy")
        metrics_beam, _ = evaluate_model(model, decoding="beam")

        avg_loss = sum(losses) / len(losses)
        print(f"Epoch {epoch + 1}: avg_train_loss={avg_loss:.3f}, final_train_loss={losses[-1]:.3f}, "
              f"greedy_bleu={metrics_greedy['bleu']:.2f}, beam_bleu={metrics_beam['bleu']:.2f}")

        epoch_results[f"epoch{epoch + 1}"] = {
            "avg_train_loss": avg_loss,
            "final_train_loss": losses[-1],
            "greedy": metrics_greedy,
            "beam": metrics_beam,
        }
        model.save_pretrained(os.path.join(OUTPUT_DIR, f"config_F_epoch{epoch + 1}_checkpoint"))

    return epoch_results

### 5. Run it: 3 epochs, full data, discriminative LR

In [ ]:
t0 = time.time()
epoch_results = train_with_checkpoints(
    lr_vision=5e-6, lr_decoder=5e-5, num_epochs=3, unfreeze_last_n_vision_layers=2,
)
print(f"Total time: {time.time() - t0:.1f}s")

rows = []
for epoch_name, r in epoch_results.items():
    for decoding in ["greedy", "beam"]:
        rows.append({
            "config": f"F_{epoch_name}_{decoding}", "epoch": epoch_name, "decoding": decoding,
            "avg_train_loss": r["avg_train_loss"], "final_train_loss": r["final_train_loss"],
            **r[decoding],
        })
results_df = pd.DataFrame(rows).set_index("config")
results_df

### 6. Save results for download

In [ ]:
results_df.to_csv(os.path.join(OUTPUT_DIR, "week3_finetune_fulldata_results.csv"))
with open(os.path.join(OUTPUT_DIR, "week3_finetune_fulldata_results.json"), "w") as f:
    json.dump(json.loads(results_df.reset_index().to_json(orient="records")), f, indent=2)

best_row = results_df["bleu"].astype(float).idxmax()
print(f"Best config by BLEU: {best_row}")
print(f"Compare: config A=27.2, config C=28.9, config D=23.4, config E=24.6 (all BLEU)")
print()
print("Download week3_finetune_fulldata_results.json/csv from the Output tab, plus ONLY the")
print(f"checkpoint folder matching the best epoch (e.g. config_F_{best_row.split('_')[1]}_checkpoint/)")
print("into this repo's results/ folder.")